## 01

### LSTM의 한계
- RNN의 약점을 보완했지만 문장이 길어지면 여전히 문제가 생김
- 문제 : gate를 거치며 오래된 종보가 점점 희석됨   
"무엇이 정말 중요한지" 강하게 선택해주지 못함

### Attention
- 핵심 단어에 더 높은 주의를 줌
- 중요한 단어 : "직원의 친절함", "기분 좋게" -> 감정 판단에 결정적
- 덜 중요 단어 : "음식은 평범했지만" -> 보조 정보 

### LSTM vs Attention
- LSTM : 문장 전체를 처음부터 끝까지 순서대로 읽음  
=> 오래된 정보가 점점 흐려짐
- Attention : "지금 어떤 단어를 더 볼지" 매 순간 선택  
=> 긴 문장에서도 중요한 정보가 사라지지 않음

### Query, Key, Value
- Query (Q) : 내가 지금 찾고 싶은 것 ("한국어 문법" 책을 찾는 내 요청)
- Key (K)   : 각 정보가 가진 표지판 (각 책 등에 붙은 제목, 분류 번호)
- Value (V) : 실제로 가져갈 내용 (책의 실제 내용)

1. Q와 K의 유사도(점수) 계산  
   score(q, kᵢ) = q · kᵢ
2. softmax로 중요도(가중치) 변환  
   αᵢ = exp(score) / Σ exp(score)  → 합이 1
3. V를 가중합  
   output = Σ αᵢ × vᵢ

### Attention 수식 : 가중 평균
- 중요한 것에 더 큰 비중을 두는 가중 평균
- Attention(Q, K, V) = softmax( QKᵀ / √dₖ ) × V
1. QKᵀ       → Q와 K의 유사도 계산 (점수)
2. / √dₖ     → 점수 크기 조정
3. softmax() → 중요도를 확률처럼 정규화 (합=1)
4. × V       → 중요도에 따라 V를 섞음

- 왜 √dₖ 로 나누나요?
- 차원(dₖ)이 커질수록 내적값이 너무 커짐  
→ softmax가 한쪽으로 쏠려 학습 불안정

### Self-Attention
- 일반 Attention은 Q가 외부에서 옴.
- Self-Attention은 Q, K, V 모두 같은 문장에서 만듬 

- 왜 강력한가?
    - RNN / LSTM : 순서대로 한 칸씩 처리 -> 먼 단어 관계를 잡기 어려움
    - Self-Attention : 모든 단어가 한꺼번에 다른 모든 단어를 봄 -> 먼 거리 관계도 직접 연결 가능

### Transformer
- Transformer는 Attention으로 문장 전체를 한 번에 바라봄
1. 문장 전체를 동시에 볼 수 있음 → 병렬 계산 가능
2. Self-Attention으로 단어 관계를 직접 계산
3. 긴 거리 의존성을 더 잘 잡음
4. 학습 속도가 빠름

## 02

In [7]:
from typing import List, Dict, Sequence, Iterable, Optional
# ==== 불용어 목록 & 텍스트 정제 ====
DEFAULT_STOPWORDS = {
    "이", "그", "저", "것", "수", "등", "들", "좀", "정말", "너무", "그리고",
    "하지만", "또", "더", "가장", "매우", "그냥", "아주", "진짜", "약간",
}
# DEFAULT_STOPWORDS가 list가 아닌 set인 이유
# 나중에 "token in stopword_set" 으로 검색할 때
# list → O(n) : 하나씩 순회해서 찾음
# set  → O(1) : 해시로 바로 찾음 → 훨씬 빠름!

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
        # 결측값이면 빈 문자열, 아니면 str()로 강제 변환
        # 데이터에 NaN 섞여있어도 에러 없이 처리
    text = re.sub(r"[^가-힣\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def keep_korean_only(text: str) -> str:
    text = re.sub(r"[^가-힣\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()
# clean_text vs keep_korean_only
# clean_text       → NaN 처리 포함 (DataFrame 입력용)
# keep_korean_only → 순수 정제만 (clean_text 이후 추가 정제용)
# 사실상 중복 나중에 추가 문자열 입력할 때 사용할 수도?

# ==== 토큰화 & 불용어 제거 ====
def tokenize(text: str) -> List[str]:
    text = text.strip()         # 앞,뒤 공백 제거 / if not text 체크를 위해 strip해줌
    if not text:
        return []
    return text.split()         # 공백 기준 분리

def remove_stopwords(tokens: Sequence[str],
                    stopwords: Optional[Iterable[str]] = None) -> List[str]:
    stopword_set = set(DEFAULT_STOPWORDS if stopwords is None else stopwords)
    return [t for t in tokens if t not in stopword_set and t.strip()]

# ==== vocab 생성 & 정수 인코딩 ====
def build_vocab(token_lists, min_freq=1, special_tokens=None):
    if special_tokens is None:
        special_tokens = ["<pad>", "<unk>"]
        # <pad> → 인덱스 0 (패딩은 항상 0이어야 Embedding의 padding_idx=0 과 맞음)
        # <unk> → 인덱스 1 (모르는 단어는 1로 처리)

    freq: Dict[str, int] = {}   # 빈 딕셔너리 생성
    for tokens in token_lists:
        for token in tokens:
            freq[token] = freq.get(token, 0) + 1
        
    vocab: Dict[str, int] = {}
    for token in special_tokens:
        vocab[token] = len(vocab)  # => vocab['<pad>'] = 0, vocab['<unk>'] = 1

    for token, count in sorted(freq.items(), key=lambda x: (-x[1], x[0])):
        # x[1] 빈도, -x[1] = 빈도 높은순 정렬
        # x[0] = 단어면, 빈도 같으면 알파벳 순
        if count >= min_freq and token not in vocab:
            vocab[token] = len(vocab)
        
    return vocab

def tokens_to_ids(tokens, vocab):
    unk_idx = vocab.get("<unk>", 1)
    return [vocab.get(token, unk_idx) for token in tokens]

# === Padding ===
def pad_sequence(ids, max_len, pad_idx=0):
    ids = list(ids[:max_len])
    if len(ids) < max_len:
        ids.extend([pad_idx] * (max_len - len(ids)))
    return ids


In [ ]:
#==== 전체 전처리 파이프라인 (preprocess_dataframe)
def preprocess_dataframe(df, text_col="review", label_col = 'sentiment', 
                         max_len=50, stopwords=None, min_freq=1):
    